# LOCO YOLO11n domain-generalization experiment

This notebook fine-tunes a COCO-pretrained YOLO11 nano model using only subsets 2, 3, and 5. Random grayscale, blur, sharpening, exposure, colour, mosaic, and geometric augmentation target warehouse domain shift. Subsets 1 and 4 remain excluded until the final locked evaluation.

## Confirm the GPU runtime

In [ ]:
import base64
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU"
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")

## Mount persistent storage and clone or update the private repository

The paid Google account must have an `inmindAcademyDetector` shortcut in My Drive pointing to the original account's shared project folder. Add a Colab secret named `GITHUB_TOKEN` with access to the private repository.

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")
STORAGE = Path("/content/drive/MyDrive/inmindAcademyDetector")
DATA_ROOT = STORAGE / "dataset"
RUNS_ROOT = STORAGE / "runs"
CONFIG_ROOT = STORAGE / "configs"
YOLO_WORKSPACE = Path("/content/loco-yolo")
for directory in (RUNS_ROOT, CONFIG_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

required_annotation = DATA_ROOT / "rgb" / "loco-sub2-v1-train.json"
assert required_annotation.is_file(), (
    "The shared Drive shortcut is not exposing the existing LOCO dataset. "
    "Do not download another copy; fix the shortcut first."
)

REPO_URL = "https://github.com/JonathanAchkouti/inmindAcademyDetector.git"
BRANCH = "codex/loco-efficient-detector"
REPO_DIR = Path("/content/inmindAcademyDetector")
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Add GITHUB_TOKEN to Colab Secrets and enable notebook access")
credential = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_environment = os.environ.copy()
git_environment.update(
    {
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.extraHeader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {credential}",
    }
)
if REPO_DIR.exists():
    subprocess.run(
        ["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, env=git_environment, check=True
    )
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", BRANCH],
        cwd=REPO_DIR,
        env=git_environment,
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        env=git_environment,
        check=True,
    )
os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
print("Dataset:", DATA_ROOT)

## Install dependencies and verify the repository

In [ ]:
%pip install -q \
    "ultralytics>=8.4,<9" \
    "albumentations>=2,<3" \
    einops "torchmetrics[detection]" pyyaml wandb ruff pytest
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
subprocess.run([sys.executable, "-m", "ruff", "format", "--check", "."], check=True)
subprocess.run([sys.executable, "-m", "ruff", "check", "."], check=True)

## Prepare the persistent E11 configuration

In [ ]:
import yaml

from utils.config import load_config

source_config = Path("configs/e11_yolo11n_domain_aug.yaml")
config = load_config(source_config)
config["data"]["raw_dir"] = str(DATA_ROOT)
config["data"]["yolo_workspace"] = str(YOLO_WORKSPACE)
config["output_dir"] = str(RUNS_ROOT / "e11_yolo11n_domain_aug")
CONFIG_PATH = CONFIG_ROOT / "e11_yolo11n_domain_aug.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
print(CONFIG_PATH.read_text())

## Build and audit the leak-free YOLO dataset view

This copies only permitted development images to the runtime's fast temporary disk; checkpoints remain in Drive. The manifest must contain only train and validation entries from subsets 2, 3, and 5.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "yolo_data.py",
        "--raw-dir",
        str(DATA_ROOT),
        "--output-dir",
        str(YOLO_WORKSPACE),
        "--copy-images",
    ],
    check=True,
)
manifest = json.loads((YOLO_WORKSPACE / "split_manifest.json").read_text())
assert manifest["include_test"] is False
assert set(manifest["subsets"]) == {"2", "3", "5"}
assert set(manifest["splits"]) == {"train", "validation"}
print(json.dumps(manifest, indent=2, sort_keys=True))

## Preview randomized domain augmentations

These are training transformations, not model predictions. Each preview is generated independently from one permitted development image.

In [ ]:
import albumentations as A
import matplotlib.pyplot as plt
import numpy as np

from dataset import LocoDataset
from yolo_augmentations import build_domain_augmentations

preview_dataset = LocoDataset(DATA_ROOT, split="train")
image_CHW, _ = preview_dataset[0]
image = image_CHW.permute(1, 2, 0).mul(255).byte().numpy()
augmentation = A.Compose(build_domain_augmentations(config["train"]["domain_augmentations"]))
figure, axes = plt.subplots(1, 5, figsize=(20, 5))
axes[0].imshow(image)
axes[0].set_title("original")
for index, axis in enumerate(axes[1:], start=1):
    axis.imshow(augmentation(image=np.asarray(image))["image"])
    axis.set_title(f"random view {index}")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

## Optional one-epoch smoke test

Leave this disabled for the real run. The smoke output is isolated and cannot overwrite E11.

In [ ]:
RUN_SMOKE_TEST = False
if RUN_SMOKE_TEST:
    subprocess.run(
        [sys.executable, "-u", "train_yolo.py", "--config", str(CONFIG_PATH), "--smoke"], check=True
    )
else:
    print("Smoke test skipped.")

## Train or resume E11

Re-running this cell resumes from `weights/last.pt`. Add the optional `DISCORD_WEBHOOK_URL` Colab secret to receive a phone notification.

In [ ]:
import requests

webhook_url = userdata.get("DISCORD_WEBHOOK_URL")


def notify(message):
    if not webhook_url:
        return
    response = requests.post(webhook_url, json={"content": message}, timeout=15)
    response.raise_for_status()


try:
    subprocess.run(
        [
            sys.executable,
            "-u",
            "train_yolo.py",
            "--config",
            str(CONFIG_PATH),
        ],
        check=True,
    )
except Exception as error:
    notify(f"LOCO E11 training failed: {type(error).__name__}")
    raise
else:
    run_dir = Path(config["output_dir"])
    summary = json.loads((run_dir / "experiment_summary.json").read_text())
    notify(f"LOCO E11 training finished!\nBest weights: {summary['best_weights']}")
    print(json.dumps(summary, indent=2, sort_keys=True))

## Official validation metric on subsets 2, 3, and 5

In [ ]:
BEST_WEIGHTS = Path(config["output_dir"]) / "weights" / "best.pt"
assert BEST_WEIGHTS.is_file(), f"Missing {BEST_WEIGHTS}"
subprocess.run(
    [
        sys.executable,
        "-u",
        "eval_yolo.py",
        "--config",
        str(CONFIG_PATH),
        "--weights",
        str(BEST_WEIGHTS),
        "--split",
        "validation",
        "--class-metrics",
    ],
    check=True,
)

## Locked final test — keep disabled during development

Set this to `True` only after the model and all settings have been selected using validation. Do not use subsets 1 and 4 for tuning.

In [ ]:
RUN_LOCKED_TEST = False
if not RUN_LOCKED_TEST:
    print("Locked test not run. This is correct during development.")
else:
    subprocess.run(
        [
            sys.executable,
            "-u",
            "eval_yolo.py",
            "--config",
            str(CONFIG_PATH),
            "--weights",
            str(BEST_WEIGHTS),
            "--split",
            "test",
            "--class-metrics",
        ],
        check=True,
    )